# AlphaI × Polaris Challenge Solution
This notebook implements the Bitcoin price prediction model using Geometric Brownian Motion (GBM) with a Student-t distribution and Dynamic Volatility Regime Adjustment.

In [ ]:
import requests
import pandas as pd
import numpy as np
from scipy.stats import t
import plotly.graph_objects as go
from datetime import datetime, timedelta
import json

## 1. Data Fetching
We use the public Binance Vision API to fetch 1-hour BTC/USDT klines.

In [ ]:
def fetch_binance_data(symbol="BTCUSDT", interval="1h", limit=1000):
    url = "https://data-api.binance.vision/api/v3/klines"
    params = {
        "symbol": symbol,
        "interval": interval,
        "limit": limit
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    
    df = pd.DataFrame(data, columns=[
        "timestamp", "open", "high", "low", "close", "volume",
        "close_time", "quote_volume", "trades", "taker_base", "taker_quote", "ignore"
    ])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit='ms')
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)
    
    return df[["timestamp", "open", "high", "low", "close", "volume"]].sort_values("timestamp").reset_index(drop=True)

## 2. Modeling
Implementing GBM + Student-t distribution + Dynamic Volatility Regime Adjustment.

In [ ]:
def predict_next_hour(data_close, short_window=10, long_window=50, num_simulations=10000, df_t=4):
    prices = np.array(data_close)
    if len(prices) < 2: return prices[-1], prices[-1]
    
    log_returns = np.log(prices[1:] / prices[:-1])
    drift = np.mean(log_returns)
    volatility = np.std(log_returns, ddof=1) if len(log_returns) > 1 else 0.0
    
    # Dynamic Volatility Regime Adjustment
    if volatility > 0 and len(prices) > long_window:
        short_vol = np.std(log_returns[-short_window:], ddof=1)
        long_vol = np.std(log_returns[-long_window:], ddof=1)
        if long_vol > 0:
            vol_ratio = np.clip(short_vol / long_vol, 0.5, 2.0)
            adj_vol = volatility * vol_ratio
        else: adj_vol = volatility
    else: adj_vol = volatility
    
    # Simulate with Student-t
    std_t = np.sqrt(df_t / (df_t - 2)) if df_t > 2 else 1.0
    random_shocks = t.rvs(df=df_t, size=num_simulations) / std_t
    simulated_returns = drift - 0.5 * (adj_vol ** 2) + adj_vol * random_shocks
    
    simulated_prices = prices[-1] * np.exp(simulated_returns)
    return float(np.percentile(simulated_prices, 2.5)), float(np.percentile(simulated_prices, 97.5))

## 3. Evaluation Metrics
Implementing Coverage, Average Width, and Winkler Score.

In [ ]:
def calculate_metrics(results, alpha=0.05):
    coverage_count = 0
    total_width = 0
    total_winkler = 0
    n = len(results)
    
    for res in results:
        L, U, Y = res['lower'], res['upper'], res['actual']
        width = U - L
        total_width += width
        if L <= Y <= U:
            coverage_count += 1
            winkler = width
        elif Y < L: winkler = width + (2 / alpha) * (L - Y)
        else: winkler = width + (2 / alpha) * (Y - U)
        total_winkler += winkler
        
    return coverage_count/n, total_width/n, total_winkler/n if n > 0 else (0,0,0)

## 4. Run 30-Day Backtest

In [ ]:
df = fetch_binance_data(limit=820) # ~30 days + warmup
prices = df['close'].values
results = []

for i in range(100, len(prices) - 1):
    lower, upper = predict_next_hour(prices[:i+1])
    results.append({"lower": lower, "upper": upper, "actual": float(prices[i+1])})

cov, width, wink = calculate_metrics(results)
print(f"Backtest Results:\nCoverage: {cov:.4f}\nAvg Width: {width:.2f}\nWinkler Score: {wink:.2f}")

# Save results
with open("backtest_results.jsonl", "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")